# Metrics for evaluating data (quality & drift)

_Metrics & Evaluation — measuring models, data & statistics_

**Before you score a model, score the data feeding it — is it clean, and does it still look like training?**

Every metric here turns a property of data into one number you can threshold.

       Quality metrics describe a single dataset: how much is missing, how much is repeated, how spread-out the categories are, how lopsided the labels are, how many rows are outliers.

---

This notebook is a practice scaffold for the **Metrics for evaluating data (quality & drift)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q evidently
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scipy + numpy

In [ ]:
# pip install scipy numpy scikit-learn evidently
import numpy as np
from scipy.stats import ks_2samp, anderson_ksamp, chi2_contingency, wasserstein_distance
from scipy.spatial.distance import mahalanobis

# ---------------------------------------------------------------
# 0) DATA-QUALITY measures on a single dataset (numpy).
# ---------------------------------------------------------------
def missingness_rate(col):          # fraction of empty cells
    return float(np.mean(np.isnan(col)))

def duplicate_rate(rows):           # fraction of repeated rows
    uniq = np.unique(rows, axis=0)
    return 1.0 - len(uniq) / len(rows)

def cardinality(col):               # number of distinct values
    return int(len(np.unique(col[~np.isnan(col)])))

def class_balance(labels):          # share of each label
    vals, counts = np.unique(labels, return_counts=True)
    return dict(zip(vals.tolist(), (counts / counts.sum()).round(3).tolist()))

def outlier_rate_z(col, cut=3.0):   # |z-score| > cut
    z = (col - col.mean()) / col.std()
    return float(np.mean(np.abs(z) > cut))

def outlier_rate_iqr(col, k=1.5):   # outside [Q1 - k*IQR, Q3 + k*IQR]
    q1, q3 = np.percentile(col, [25, 75]); iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return float(np.mean((col < lo) | (col > hi)))

# ---------------------------------------------------------------
# 1) PSI / CSI from scratch (same formula; CSI = PSI on a feature).
# ---------------------------------------------------------------
def psi(ref, cur, bins=10):
    q = np.quantile(ref, np.linspace(0, 1, bins + 1))   # quantile edges from REFERENCE
    q[0], q[-1] = -np.inf, np.inf
    e = np.histogram(ref, q)[0] / len(ref)              # expected share per bin
    a = np.histogram(cur, q)[0] / len(cur)              # actual share per bin
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))       # sum (a_i - e_i) * ln(a_i / e_i)
# rule of thumb: <0.1 stable, 0.1-0.25 moderate, >0.25 large drift -> alarm

# ---------------------------------------------------------------
# 2) Two-sample tests for NUMERIC drift (reference vs current).
# ---------------------------------------------------------------
ref = np.random.RandomState(0).normal(0, 1, 2000)       # placeholder samples
cur = np.random.RandomState(1).normal(0.3, 1, 2000)

ks = ks_2samp(ref, cur)                                  # Kolmogorov-Smirnov: max CDF gap
print("KS D =", round(ks.statistic, 3), " p =", ks.pvalue)
ad = anderson_ksamp([ref, cur])                          # Anderson-Darling (tail-sensitive)
print("AD stat =", round(ad.statistic, 3))
print("Wasserstein =", round(wasserstein_distance(ref, cur), 3))  # Earth Mover's distance
# (Cramer-von Mises: scipy.stats.cramervonmises_2samp(ref, cur) -- integral of squared CDF gap)

# ---------------------------------------------------------------
# 3) CATEGORICAL drift: chi-square on a contingency table.
# ---------------------------------------------------------------
#            cat A  cat B  cat C
table = np.array([[120,  80,  40],     # reference counts
                  [ 90,  70,  90]])    # current counts
chi2, p, dof, _ = chi2_contingency(table)
print("chi-square =", round(chi2, 2), " p =", round(p, 4))

# ---------------------------------------------------------------
# 4) MULTIVARIATE outlier: Mahalanobis distance.
# ---------------------------------------------------------------
X = np.random.RandomState(2).multivariate_normal([0, 0], [[1, .8], [.8, 1]], 500)
mu = X.mean(axis=0); VI = np.linalg.inv(np.cov(X.T))     # inverse covariance
d = np.array([mahalanobis(row, mu, VI) for row in X])    # per-row distance
print("multivariate outlier rate (d > 3):", float(np.mean(d > 3)))

# ---------------------------------------------------------------
# 5) EVIDENTLY packages all of the above into one drift report.
# ---------------------------------------------------------------
# from evidently.report import Report
# from evidently.metric_preset import DataDriftPreset
# rep = Report(metrics=[DataDriftPreset()])     # PSI / KS / Wasserstein per column
# rep.run(reference_data=ref_df, current_data=cur_df)
# rep.save_html("drift_dashboard.html")

## Visualize the data & results

_Splitting load_wine in half (rows are ordered by cultivar, so the halves are genuinely different wines), which features drift, and which cross the PSI 0.25 alarm line?_

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

d = load_wine()                                  # 178 wines, 13 features, ordered by cultivar
X, names = d.data, list(d.feature_names)

half = len(X) // 2                               # 89
ref, cur = X[:half], X[half:]                    # halves are genuinely different cultivars

def psi(r, c, bins=10):
    q = np.quantile(r, np.linspace(0, 1, bins + 1)); q[0], q[-1] = -np.inf, np.inf
    e = np.histogram(r, q)[0] / len(r)
    a = np.histogram(c, q)[0] / len(c)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))

scores = {nm: round(psi(ref[:, i], cur[:, i]), 3) for i, nm in enumerate(names)}
for nm, p in sorted(scores.items(), key=lambda t: -t[1]):
    print(f"{nm:32s} {p}")
# proline 4.283, alcalinity_of_ash 3.318, hue 2.379, flavanoids 2.361,
# total_phenols 1.852, ... color_intensity 0.824, ... ash 0.190
# alarm line at 0.25: every feature except 'ash' is flagged as large drift.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A numeric feature is monitored daily. Reference bin shares $e=[0.5,0.3,0.2]$; today's sample $a=[0.3,0.3,0.4]$. Compute the PSI and classify it on the <0.1 / 0.1&ndash;0.25 / >0.25 scale.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bin 1: $(0.3-0.5)\ln(0.3/0.5)=(-0.2)(-0.511)=0.102$. — _Both factors negative &rarr; a positive contribution; this bin lost a lot of mass._
- Bin 2: $(0.3-0.3)\ln(0.3/0.3)=0$. — _No change in a bin contributes nothing._
- Bin 3: $(0.4-0.2)\ln(0.4/0.2)=(0.2)(0.693)=0.139$. — _This bin gained mass; both factors positive._
- Sum: $0.102+0+0.139=0.241$. — _PSI adds the per-bin contributions._

**Answer:** PSI $=0.241$, which lands in the 0.1&ndash;0.25 moderate band (just under the 0.25 alarm line). Mass clearly slid from bin 1 to bin 3, so you would flag it for watching and confirm whether it actually hurts the model before retraining.

</details>

**Problem 2.** Your KS test on a feature returns $D=0.004$ with $p=2\times10^{-8}$ across 5 million rows. The dashboard screams "significant drift". Should you alert? What metric would you trust instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Separate significance from size: the p-value says the shift is real, not that it is big. — _Any tiny difference becomes "significant" once $N$ is huge — the test is over-powered._
- Read the effect size $D=0.004$: the biggest CDF gap is 0.4%. — _$D$ is a 0-to-1 distance independent of sample size; 0.004 is negligible._
- Threshold an effect size (KS $D$, PSI, or Wasserstein), or subsample to a few thousand rows before testing. — _Effect sizes measure practical drift; subsampling stops $N$ from manufacturing significance._

**Answer:** Do not alert. The microscopic p-value is an artifact of 5 million rows — KS is over-powered at that scale. The honest signal is the effect size: $D=0.004$ (a 0.4% CDF gap) or a PSI well under 0.1 means the distribution barely moved. Threshold the effect size, not the p-value (or subsample first).

</details>

**Problem 3.** A row has a perfectly normal height (170 cm) and a perfectly normal weight (45 kg), yet it is clearly anomalous. Which quality metric catches it, and why do per-column z-scores miss it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check each column alone: 170 cm and 45 kg are each within normal range, so $|z|$ is small for both. — _A z-score looks at one column at a time and sees nothing wrong._
- Note that height and weight are strongly correlated — tall-and-very-light is the odd combination. — _The anomaly lives in the relationship between columns, not in any single one._
- Compute the Mahalanobis distance, which uses $\Sigma^{-1}$ to account for that correlation. — _It measures distance from the center in the whitened space where correlations are removed, so the off-trend combination scores far._

**Answer:** Use Mahalanobis distance $d_M=\sqrt{(\mathbf{x}-\boldsymbol{\mu})^{\top}\Sigma^{-1}(\mathbf{x}-\boldsymbol{\mu})}$. Per-column z-scores treat columns independently, so "normal height, normal weight" passes both. Mahalanobis folds in the height&ndash;weight correlation via $\Sigma^{-1}$, so a tall-yet-very-light row sits far from the joint center and is flagged as a multivariate outlier.

</details>